# Demo A: Feel the Pain

**Workshop Opening Demo** | LLM Inference at Scale

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/workshop/demos/demo_a_feel_the_pain.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/workshop/demos/demo_a_feel_the_pain.ipynb)

**Goal:** Load a real 7B model on GPU, measure what actually happens, and expose
three problems engineers hit in production: memory consumption, slow first token,
and throughput ceiling.

**Requirements:** GPU with >= 16 GB VRAM (Molab RTX Pro 6000, A100, etc.)

In [ ]:
# Install dependencies
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'transformers', 'torch', 'matplotlib', 'accelerate'])

# Imports
import torch
import time
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Config
MODEL_ID = 'mistralai/Mistral-7B-v0.1'  # non-gated, no auth needed
DEVICE = 'cuda'
DTYPE = torch.float16

# GPU info
gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name} ({gpu_mem_gb:.1f} GB)')
print(f'Precision: FP16')
print(f'Model: {MODEL_ID}')

## Problem 1: Memory Consumption

A 7B parameter model in FP16 should theoretically need ~14.5 GB.
Let's load it and measure the actual GPU memory consumed.

In [ ]:
# Measure GPU memory before and after loading
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
mem_before = torch.cuda.memory_allocated() / 1e9

print('Loading model... (30-60 seconds)')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=DTYPE, device_map='auto', token=False)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=False)

torch.cuda.synchronize()
model_mem_gb = torch.cuda.memory_allocated() / 1e9 - mem_before

print(f'\n--- MODEL LOADED ---')
print(f'GPU memory used by model: {model_mem_gb:.2f} GB')
print(f'This is JUST the weights. No users served yet.')
print(f'On a {gpu_mem_gb:.0f} GB GPU, you have {gpu_mem_gb - model_mem_gb:.1f} GB left for everything else.')

## Memory Growth increases With Every Token

Each token generated adds to the memory. Let's measure actual GPU memory
at different generation lengths — this is real `torch.cuda` measurement, not a formula.

In [ ]:
# Measure REAL GPU memory at increasing generation lengths
# Use longer sequences to make KV growth visible on fast GPUs
GEN_LENGTHS = [0, 128, 256, 512, 1024]
mem_inp = tokenizer('Explain the history of AI from 1950 to today: ' * 50,
                    return_tensors='pt', truncation=True, max_length=512).to(DEVICE)
n_mem_input = mem_inp['input_ids'].shape[1]

# Warmup at max length
with torch.no_grad():
    model.generate(**mem_inp, max_new_tokens=GEN_LENGTHS[-1], do_sample=False, pad_token_id=tokenizer.eos_token_id)
torch.cuda.empty_cache()

mem_gb = []
for gen_len in GEN_LENGTHS:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    if gen_len == 0:
        mem_gb.append(torch.cuda.memory_allocated() / 1e9)
    else:
        with torch.no_grad():
            model.generate(**mem_inp, max_new_tokens=gen_len, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        torch.cuda.synchronize()
        mem_gb.append(torch.cuda.max_memory_allocated() / 1e9)
    print(f'  {n_mem_input + gen_len:>5} total tokens -> {mem_gb[-1]:.2f} GB')

kv_growth = mem_gb[-1] - mem_gb[0]
print(f'\nKV cache grew by {kv_growth:.2f} GB for ONE user ({n_mem_input + GEN_LENGTHS[-1]} tokens)')
print(f'This is real measured growth, not a formula.')
print(f'80 users at this context: ~{kv_growth * 80:.0f} GB — exceeds any single GPU.')


In [ ]:
# Plot memory growth (delta from baseline) — this is the cost PER USER
baseline = mem_gb[0]
kv_delta_gb = [m - baseline for m in mem_gb]

fig_mem, ax_mem = plt.subplots(figsize=(9, 4))
mem_labels = [f'{n_mem_input + gl}' for gl in GEN_LENGTHS]
bar_colors = ['#f3f4f6', '#dbeafe', '#dbeafe', '#fef3c7', '#ffe4e6']

ax_mem.bar(range(len(GEN_LENGTHS)), kv_delta_gb, color=bar_colors, edgecolor='#000', linewidth=1.2)
ax_mem.set_xticks(range(len(GEN_LENGTHS)))
ax_mem.set_xticklabels([f'{l} tokens' for l in mem_labels])
ax_mem.set_ylabel('KV Cache Growth (GB)', fontsize=11)
ax_mem.set_title('KV Cache Memory Cost Per User (grows with context)', fontsize=12, fontweight='bold')
for mi, mv in enumerate(kv_delta_gb):
    ax_mem.text(mi, mv + max(kv_delta_gb) * 0.02, f'{mv:.2f} GB', ha='center', fontsize=10)
ax_mem.spines['top'].set_visible(False)
ax_mem.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


## Problem 2: Slow First Token (TTFT)

Time to First Token is the latency before the user sees ANY response.
Let's measure it with a simple prompt.

In [ ]:
# Measure TTFT with a short prompt
WARMUP_PROMPT = "Hello World! This is the AIE World Fair 2026"
TTFT_PROMPT = 'Explain quantum computing in simple terms'

warmup_inputs = tokenizer(WARMUP_PROMPT, return_tensors='pt').to(DEVICE)
ttft_inputs = tokenizer(TTFT_PROMPT, return_tensors='pt').to(DEVICE)
ttft_n_tokens = ttft_inputs['input_ids'].shape[1]

# Warm up (first call compiles CUDA kernels)
with torch.no_grad():
    model.generate(**warmup_inputs, max_new_tokens=1, pad_token_id=tokenizer.eos_token_id)

# Actual measurement
torch.cuda.synchronize()
ttft_t0 = time.perf_counter()
with torch.no_grad():
    model.generate(**ttft_inputs, max_new_tokens=1, pad_token_id=tokenizer.eos_token_id)
torch.cuda.synchronize()
ttft_ms = (time.perf_counter() - ttft_t0) * 1000

print(f'Prompt: "{TTFT_PROMPT}"')
print(f'Input tokens: {ttft_n_tokens}')
print(f'Time to First Token: {ttft_ms:.0f} ms')
print(f'\nThis is fast because the prompt is short ({ttft_n_tokens} tokens).')
print(f'Watch what happens when we scale to thousands of tokens...')


## Longer Context = Slower TTFT

The model must process ALL input tokens before generating the first output.
Let's measure TTFT at different prompt lengths to see this "prefill" cost.

In [ ]:
# Measure TTFT at increasing prompt lengths
BASE_TEXT = 'The quick brown fox jumps over the lazy dog. '
TARGET_LENGTHS = [128, 1024, 4096, 8192, 16384]

# Warmup at longest length so memory allocation doesn't skew timing
warmup_inp = tokenizer(BASE_TEXT * (TARGET_LENGTHS[-1] // 10), return_tensors='pt',
                       truncation=True, max_length=TARGET_LENGTHS[-1]).to(DEVICE)
with torch.no_grad():
    model.generate(**warmup_inp, max_new_tokens=1, pad_token_id=tokenizer.eos_token_id)
del warmup_inp
torch.cuda.empty_cache()

# Measure at each length
ttft_results = []
for target in TARGET_LENGTHS:
    scaling_inp = tokenizer(BASE_TEXT * (target // 10), return_tensors='pt',
                            truncation=True, max_length=target).to(DEVICE)
    scaling_n = scaling_inp['input_ids'].shape[1]

    torch.cuda.synchronize()
    scaling_t0 = time.perf_counter()
    with torch.no_grad():
        model.generate(**scaling_inp, max_new_tokens=1, pad_token_id=tokenizer.eos_token_id)
    torch.cuda.synchronize()
    scaling_ms = (time.perf_counter() - scaling_t0) * 1000

    ttft_results.append((scaling_n, scaling_ms))
    print(f'  {scaling_n:>5} tokens -> TTFT: {scaling_ms:>7.1f} ms')
    del scaling_inp

print(f'\nLonger prompts = longer wait. This is the prefill cost.')
print(f'At {ttft_results[-1][0]} tokens, TTFT is {ttft_results[-1][1]/ttft_results[0][1]:.1f}x slower than at {ttft_results[0][0]} tokens.')


In [ ]:
# Visualize TTFT vs prompt length
fig_ttft_chart, ax_ttft_chart = plt.subplots(figsize=(8, 4))
ttft_tokens = [r[0] for r in ttft_results]
ttft_times = [r[1] for r in ttft_results]

ax_ttft_chart.bar(range(len(ttft_tokens)), ttft_times, color='#dbeafe', edgecolor='#000', linewidth=1.2)
ax_ttft_chart.set_xticks(range(len(ttft_tokens)))
ax_ttft_chart.set_xticklabels([str(t) for t in ttft_tokens])
ax_ttft_chart.set_xlabel('Input Tokens', fontsize=11)
ax_ttft_chart.set_ylabel('TTFT (ms)', fontsize=11)
ax_ttft_chart.set_title('Time to First Token vs Prompt Length', fontsize=12, fontweight='bold')
for ti, tv in enumerate(ttft_times):
    ax_ttft_chart.text(ti, tv + max(ttft_times) * 0.02, f'{tv:.0f}ms', ha='center', fontsize=10)
ax_ttft_chart.spines['top'].set_visible(False)
ax_ttft_chart.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## Problem 3: Throughput Ceiling

With naive `generate()`, the GPU serves ONE user at a time.
Let's run 5 requests sequentially and measure the real queue delay.

In [ ]:
# Run 5 sequential requests -- this is what happens without batching
N_USERS = 5
TOKENS_PER_USER = 50
tp_inp = tokenizer('Explain how neural networks learn:', return_tensors='pt').to(DEVICE)

# Warm up
with torch.no_grad():
    model.generate(**tp_inp, max_new_tokens=1)

# Time each request
user_times = []
for _ in range(N_USERS):
    torch.cuda.synchronize()
    tp_t0 = time.perf_counter()
    with torch.no_grad():
        model.generate(**tp_inp, max_new_tokens=TOKENS_PER_USER, do_sample=False)
    torch.cuda.synchronize()
    user_times.append(time.perf_counter() - tp_t0)

total = sum(user_times)
throughput = N_USERS * TOKENS_PER_USER / total

# Print the results
print(f'=== {N_USERS} users, {TOKENS_PER_USER} tokens each (sequential, no batching) ===')
print(f'Total wall time: {total:.2f}s')
print(f'Throughput: {throughput:.1f} tok/s')
print()
print('Per-user wait time (queue + generation):')
cumulative = 0.0
for uj, ut in enumerate(user_times):
    cumulative += ut
    print(f'  User {uj+1}: response complete at {cumulative:.2f}s')
print()
print(f'--- THE PROBLEM ---')
print(f'User 1 gets response in {user_times[0]:.1f}s')
print(f'User {N_USERS} waits {total:.1f}s (queued behind everyone else)')
print(f'With 100 users: last one waits ~{total / N_USERS * 100:.0f}s')

In [ ]:
# Visualize the queue: gray = waiting, color = generating
fig_queue, ax_queue = plt.subplots(figsize=(10, 3.5))
queue_colors = ['#dbeafe', '#dcfce7', '#fef3c7', '#f3e8ff', '#ffe4e6']
queue_start = 0.0

for qi, qt in enumerate(user_times):
    if queue_start > 0:
        ax_queue.barh(qi, queue_start, color='#f3f4f6', edgecolor='#000', linewidth=0.8)
    ax_queue.barh(qi, qt, left=queue_start, color=queue_colors[qi], edgecolor='#000', linewidth=0.8)
    ax_queue.text(queue_start + qt + 0.03, qi, f'{queue_start + qt:.2f}s', va='center', fontsize=9)
    queue_start += qt

ax_queue.set_yticks(range(N_USERS))
ax_queue.set_yticklabels([f'User {k+1}' for k in range(N_USERS)])
ax_queue.set_xlabel('Time (seconds)', fontsize=11)
ax_queue.set_title(f'Sequential Queue: {N_USERS} Users, {TOKENS_PER_USER} Tokens Each (No Batching)',
             fontsize=12, fontweight='bold')
ax_queue.legend(handles=[Patch(fc='#f3f4f6', ec='#000', label='Waiting in queue'),
                    Patch(fc='#dbeafe', ec='#000', label='Generating tokens')],
          loc='lower right', fontsize=10)
ax_queue.spines['top'].set_visible(False)
ax_queue.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

wait_pct = (total - user_times[-1]) / total * 100
print(f'\nGray = idle wait. Color = GPU working on YOUR request.')
print(f'User {N_USERS} spends {wait_pct:.0f}% of their time just waiting in the queue.')

## Summary: Three Problems to Solve

| Problem | What We Measured | Why It Matters |
|---------|-----------------|----------------|
| **Memory** | ~14.5 GB for weights alone, grows with context and users | Limits how many users fit on one GPU |
| **TTFT** | Grows linearly with context | Users wait before seeing any response |
| **Throughput** | ~50 tok/s for 1 user | Queue builds up with concurrent users |

The rest of this workshop shows you:
1. **WHY** these problems exist (memory equation, roofline model)
2. **HOW** to fix them (GQA, quantization, prefix caching, eviction)
3. **WHICH TOOLS** implement the fixes (vLLM, SGLang, TensorRT-LLM)